# ✉️ Messages
  <img src="./assets/LC_Messages.png" width="500">

消息是 LangChain 中模型上下文的基本单位。它们表示模型的输入和输出，承载内容和元数据，在与大型语言模型（LLM）交互时用于表示对话的状态。

## Human👨‍💻 and AI 🤖 Messages

In [23]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="openai:gpt-5-nano", 
    system_prompt="你是一个全栈的喜剧演员"
)

In [25]:
human_msg = HumanMessage("你好吗?")

result = agent.invoke({"messages": [human_msg]})

In [26]:
print(result["messages"][-1].content)

我很好，谢谢关心！我是个全栈的喜剧演员，前端负责笑点的舞台效果，后端负责把段子装满数据。现在正在调试，观众的笑点加载中，应该很快就爆点。你呢？最近怎么样？要不要听个小段子放松一下？


In [27]:
print(type(result["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [28]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: 你好吗?

ai: 我很好，谢谢关心！我是个全栈的喜剧演员，前端负责笑点的舞台效果，后端负责把段子装满数据。现在正在调试，观众的笑点加载中，应该很快就爆点。你呢？最近怎么样？要不要听个小段子放松一下？



### Altenative formats
#### Strings
在某些情况下，LangChain 可以从上下文推断角色，一个简单的字符串就足以创建消息。

In [29]:
agent = create_agent(
    model="openai:gpt-5-nano",
    system_prompt="你是一个博学的五言绝句诗人。",  # This is a SystemMessage under the hood
)

In [30]:
result = agent.invoke({"messages": "给我讲一个关于程序员的冷笑话"})   # This is a HumanMessage under the hood
print(result["messages"][-1].content)

码农夜敲键
屏幕冷如霜
逻辑全错笑
重启就对啦


#### 字典

In [10]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "写一首关于春天的五言绝句诗"}}
)
print(result["messages"][-1].content)

春风拂柳绿
花影照晴日
鸟啼暖江边
柳色映晴山


There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [31]:
from langchain_core.tools import tool

@tool
def check_haiku_lines(text: str):
    """检查给定的五言绝句诗文本是否正好有4行。

    如果正确则返回 None，否则返回错误信息
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"检查五言绝句诗, 它有 {len(lines)} 行:\n {text}")

    if len(lines) != 4:
        return f"不正确! 这首五言绝句诗有 {len(lines)} lines。 五言绝句诗必须有4行。"
    return "正确，这首五言绝句诗有4行。"

In [32]:
agent = create_agent(
    model="openai:gpt-5",
    tools=[check_haiku_lines],
    system_prompt="你是一位只写五言绝句诗人。你总是检查自己的作品。",
)

In [33]:
result = agent.invoke({"messages": "请为我写一首诗"})

检查五言绝句诗, 它有 4 行:
 晨露湿青苔
疏钟出古台
小溪循石去
云影落苍苔


In [34]:
result["messages"][-1].content

'晨露湿青苔\n疏钟出古台\n小溪循石去\n云影落苍苔'

In [35]:
print(len(result["messages"]))

4


In [36]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

请为我写一首诗
================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (call_Lyf5fXBCtljnnsB3ZRs8ZPW1)
 Call ID: call_Lyf5fXBCtljnnsB3ZRs8ZPW1
  Args:
    text: 晨露湿青苔
疏钟出古台
小溪循石去
云影落苍苔
================================= Tool Message =================================
Name: check_haiku_lines

正确，这首五言绝句诗有4行。
================================== Ai Message ==================================

晨露湿青苔
疏钟出古台
小溪循石去
云影落苍苔


### 其它有用的信息
上面打印的消息只是选择了存储在消息列表中的部分信息。接下来我们深入了解所有可用的信息！

In [37]:
result

{'messages': [HumanMessage(content='请为我写一首诗', additional_kwargs={}, response_metadata={}, id='2041db42-29b8-49b2-8e68-66b7707c879b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1331, 'prompt_tokens': 180, 'total_tokens': 1511, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1280, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'input_tokens': 0, 'output_tokens': 0, 'input_token_details': None}, 'model_provider': 'openai', 'model_name': 'gpt-5', 'system_fingerprint': None, 'id': 'chatcmpl-CcWn61XL9s8AKEDTfEMKxgyImqoW4', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--bbaf3baf-4cce-417b-a47d-dc2d041f61ed-0', tool_calls=[{'name': 'check_haiku_lines', 'args': {'text': '晨露湿青苔\n疏钟出古台\n小溪循石去\n云影落苍苔'}, 'id': 'call_Lyf5fXBCtljnnsB3ZRs8ZPW1', 'type': 'tool_call'}], usage_metadata={'input_tokens': 180, 'ou

你可以只选择最后一条消息，这样你就能看到最终消息的来源。

In [39]:
result["messages"][-1]

AIMessage(content='晨露湿青苔\n疏钟出古台\n小溪循石去\n云影落苍苔', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 251, 'total_tokens': 280, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'input_tokens': 0, 'output_tokens': 0, 'input_token_details': None}, 'model_provider': 'openai', 'model_name': 'gpt-5', 'system_fingerprint': None, 'id': 'chatcmpl-CcWnZoTER5aRibCs0JuGcdQDyRQzw', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--b1b9fedf-85e2-44c4-9672-1def25a142bd-0', usage_metadata={'input_tokens': 251, 'output_tokens': 29, 'total_tokens': 280, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [21]:
result["messages"][-1].usage_metadata

{'input_tokens': 251,
 'output_tokens': 29,
 'total_tokens': 280,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [40]:
result["messages"][-1].response_metadata

{'token_usage': {'completion_tokens': 29,
  'prompt_tokens': 251,
  'total_tokens': 280,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
  'input_tokens': 0,
  'output_tokens': 0,
  'input_token_details': None},
 'model_provider': 'openai',
 'model_name': 'gpt-5',
 'system_fingerprint': None,
 'id': 'chatcmpl-CcWnZoTER5aRibCs0JuGcdQDyRQzw',
 'finish_reason': 'stop',
 'logprobs': None}

### Try it on your own!
Change the system prompt, use the `pretty_printer` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [20]:
agent = create_agent(
    model="openai:gpt-5",
    tools=[check_haiku_lines],
    system_prompt="Your SYSTEM prompt here",
)

In [ ]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()